# FinalTest Tools

这个 Notebook 只做文件管理，不训练模型。

常用场景：
- 查看当前目录、模型、权重、预测文件是否存在
- 备份 `./deberta-v3-large`
- 从备份还原 `./deberta-v3-large`
- 删除已有 NER/RE 权重，重新训练前清场
- 打包 `predictions.json` 和输出模型，方便下载

安全规则：删除/还原类 Cell 默认不会执行，必须把对应的 `CONFIRM_...` 改成 `True`。


### Cell 1: 状态检查

In [ ]:
from pathlib import Path
import json
import os
import shutil
import time
import zipfile

ROOT = Path(".").resolve()
BASE_MODEL = ROOT / "deberta-v3-large"
BACKUP_ROOT = ROOT / "model_backups"
NER_OUT = ROOT / "output_ner"
RE_OUT = ROOT / "output_re"
PRED_FILE = ROOT / "predictions.json"

def format_size(num_bytes):
    if num_bytes is None:
        return "-"
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024

def path_size(path):
    path = Path(path)
    if not path.exists():
        return None
    if path.is_file():
        return path.stat().st_size
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total

def show_path(path):
    path = Path(path)
    status = "DIR" if path.is_dir() else "FILE" if path.is_file() else "MISSING"
    print(f"{status:7s} {str(path):35s} {format_size(path_size(path))}")

print("ROOT:", ROOT)
print()
for p in [BASE_MODEL, NER_OUT, RE_OUT, PRED_FILE, BACKUP_ROOT]:
    show_path(p)

print("\nPossible base-model backups:")
candidates = []
if BACKUP_ROOT.exists():
    candidates.extend(sorted(BACKUP_ROOT.iterdir()))
candidates.extend(sorted(ROOT.glob("deberta-v3-large_backup*")))
if candidates:
    for p in candidates:
        show_path(p)
else:
    print("  No backups found yet.")


### Cell 2: 备份基座模型

In [ ]:
# 把 ./deberta-v3-large 复制到 ./model_backups/ 下。
# 这是非破坏性操作，但会占用几 GB 磁盘空间。

CONFIRM_BACKUP_BASE_MODEL = False
BACKUP_NAME = ""  # 留空则自动用时间戳；也可以写 "deberta-v3-large_backup_a10_ok"

if not CONFIRM_BACKUP_BASE_MODEL:
    print("Not running. Set CONFIRM_BACKUP_BASE_MODEL = True to backup the base model.")
else:
    assert BASE_MODEL.exists() and BASE_MODEL.is_dir(), f"Base model folder not found: {BASE_MODEL}"
    BACKUP_ROOT.mkdir(exist_ok=True)
    name = BACKUP_NAME.strip() or f"deberta-v3-large_backup_{time.strftime('%Y%m%d_%H%M%S')}"
    target = BACKUP_ROOT / name
    assert not target.exists(), f"Backup already exists: {target}"
    print(f"Copying {BASE_MODEL} -> {target}")
    shutil.copytree(BASE_MODEL, target)
    show_path(target)
    print("Backup complete.")


### Cell 3: 从备份还原基座模型

In [ ]:
# 从某个备份还原 ./deberta-v3-large。
# 为安全起见，当前 ./deberta-v3-large 不会直接丢弃，会先挪到 *_before_restore_时间戳。

CONFIRM_RESTORE_BASE_MODEL = False
RESTORE_FROM = ""  # 例: "model_backups/deberta-v3-large_backup_20260521_120000"

if not CONFIRM_RESTORE_BASE_MODEL:
    print("Not running. Set CONFIRM_RESTORE_BASE_MODEL = True and RESTORE_FROM to restore.")
else:
    source = Path(RESTORE_FROM).expanduser()
    if not source.is_absolute():
        source = ROOT / source
    assert source.exists() and source.is_dir(), f"Backup folder not found: {source}"

    if BASE_MODEL.exists():
        parked = ROOT / f"deberta-v3-large_before_restore_{time.strftime('%Y%m%d_%H%M%S')}"
        print(f"Moving current base model to: {parked}")
        shutil.move(str(BASE_MODEL), str(parked))

    print(f"Restoring {source} -> {BASE_MODEL}")
    shutil.copytree(source, BASE_MODEL)
    show_path(BASE_MODEL)
    print("Restore complete.")


### Cell 4: 删除已有训练权重和预测文件

In [ ]:
# 清理训练输出。适合正式重跑前使用。
# 注意：删除 output_ner/output_re 会删除已经训练好的权重。

CONFIRM_DELETE_OUTPUTS = False
DELETE_NER = True
DELETE_RE = True
DELETE_PREDICTIONS = True
DELETE_BATCH_TEST_DIRS = True

targets = []
if DELETE_NER:
    targets.append(NER_OUT)
if DELETE_RE:
    targets.append(RE_OUT)
if DELETE_PREDICTIONS:
    targets.append(PRED_FILE)
if DELETE_BATCH_TEST_DIRS:
    targets.extend([ROOT / "tmp_batchtest_ner", ROOT / "tmp_batchtest_re"])

print("Targets:")
for t in targets:
    show_path(t)

if not CONFIRM_DELETE_OUTPUTS:
    print("\nNot deleting. Set CONFIRM_DELETE_OUTPUTS = True to delete these targets.")
else:
    for t in targets:
        if t.is_dir():
            shutil.rmtree(t)
            print("Deleted dir:", t)
        elif t.is_file():
            t.unlink()
            print("Deleted file:", t)
        else:
            print("Skip missing:", t)
    print("Cleanup complete.")


### Cell 5: 打包结果，方便下载

In [ ]:
# 把 predictions.json、output_ner、output_re 打成 zip。
# 如果只想提交 predictions.json，也可以只下载那个文件。

ARCHIVE_NAME = f"finaltest_outputs_{time.strftime('%Y%m%d_%H%M%S')}.zip"
archive_path = ROOT / ARCHIVE_NAME

include_paths = [PRED_FILE, NER_OUT, RE_OUT]
existing = [p for p in include_paths if p.exists()]
assert existing, "No output files found to archive."

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in existing:
        if p.is_file():
            zf.write(p, p.relative_to(ROOT))
        else:
            for child in p.rglob("*"):
                if child.is_file():
                    zf.write(child, child.relative_to(ROOT))

show_path(archive_path)
print("Archive created.")


### Cell 6: 清理 HuggingFace 坏缓存

In [ ]:
# 下载中断后，HF cache 偶尔会留下坏缓存。一般不需要跑。
# 这个 Cell 只删缓存，不删 ./deberta-v3-large。

CONFIRM_CLEAR_HF_CACHE = False
cache_targets = [
    Path("/root/.cache/huggingface/hub/models--microsoft--deberta-v3-large"),
    Path.home() / ".cache/huggingface/hub/models--microsoft--deberta-v3-large",
]

for p in cache_targets:
    show_path(p)

if not CONFIRM_CLEAR_HF_CACHE:
    print("\nNot clearing cache. Set CONFIRM_CLEAR_HF_CACHE = True to clear.")
else:
    for p in cache_targets:
        if p.exists():
            shutil.rmtree(p)
            print("Cleared:", p)
    print("Cache cleanup complete.")


### Cell 7: 当前推荐训练命令

In [ ]:
print("NER full training:")
print("python train_ner.py --data HW_train_data.json --model ./deberta-v3-large --save_dir ./output_ner --max_length 384 --batch_size 12 --epochs 8 --lr 1e-5 --warmup_ratio 0.1 --dropout 0.1 --val_ratio 0.05 --patience 2")
print()
print("RE full training:")
print("python train_re.py --data HW_train_data.json --model ./deberta-v3-large --save_dir ./output_re --max_length 384 --batch_size 12 --epochs 6 --lr 1e-5 --warmup_ratio 0.1 --dropout 0.1 --neg_ratio 1 --val_ratio 0.05 --patience 2")
print()
print("Continue RE from current best to epoch 8:")
print("python train_re.py --data HW_train_data.json --model ./deberta-v3-large --save_dir ./output_re --max_length 384 --batch_size 12 --epochs 8 --lr 1e-5 --warmup_ratio 0.1 --dropout 0.1 --neg_ratio 1 --val_ratio 0.05 --patience 2 --resume_from ./output_re/best_re_model.pt")
print()
print("Inference:")
print("python predict.py --ner_model_dir ./output_ner --re_model_dir ./output_re --data HW_train_data.json --output predictions.json --batch_size 12 --re_threshold 0.9 --max_length 384 --val_ratio 0.05 --evaluate")


### Cell 8: 继续训练 RE 到第 8 轮

In [ ]:
# 如果 RE best checkpoint 是第 6 轮，这个 Cell 会继续跑第 7-8 轮。
# 如果第 7/8 轮没有更好，旧的 best_re_model.pt 会保留。

CONFIRM_CONTINUE_RE = False
TARGET_EPOCHS = 8

if not CONFIRM_CONTINUE_RE:
    print("Not running. Set CONFIRM_CONTINUE_RE = True to continue RE training.")
else:
    import subprocess
    from pathlib import Path

    ROOT = Path(".").resolve()
    RE_OUT = ROOT / "output_re"

    assert (RE_OUT / "best_re_model.pt").exists(), "Missing ./output_re/best_re_model.pt"
    cmd = [
        "python", "train_re.py",
        "--data", "HW_train_data.json",
        "--model", "./deberta-v3-large",
        "--save_dir", "./output_re",
        "--max_length", "384",
        "--batch_size", "12",
        "--epochs", str(TARGET_EPOCHS),
        "--lr", "1e-5",
        "--warmup_ratio", "0.1",
        "--dropout", "0.1",
        "--neg_ratio", "1",
        "--val_ratio", "0.05",
        "--patience", "2",
        "--resume_from", "./output_re/best_re_model.pt",
    ]
    print("Running:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Continue RE training complete.")
